# XGB-D03 với kênh giao dịch × khung giờ 6 tiếng

Đối chứng `XGB-D03` · Lượt thử `XGB-D03-CHANNEL-6H`

## Giả thuyết từ EDA

Thêm tổ hợp kênh giao dịch và giờ trong ngày để cây có thể tách, chẳng hạn, giao dịch tại điểm bán vào ban đêm khỏi các giao dịch còn lại. Phép sàng lọc trước đó ghi nhận mức tăng OOF ROC AUC **+0.000021** ở **4/5 fold** khi hiệu chỉnh điểm dự đoán theo nhóm. Notebook này kiểm tra lại bằng cách huấn luyện XGBoost với đầu vào bổ sung.

[Xem phân tích EDA](../../ket_qua_eda.md)

## Dữ liệu và đối chứng

Dùng dữ liệu cuộc thi gồm **699.960 dòng train** và **299.984 dòng test**, cùng 5 fold đã chốt. `XGB-D03` gốc có **109 cột đầu vào** từ 24 đặc trưng gốc. Thí nghiệm giữ nguyên đầu vào gốc, seed, cách chia fold và tham số XGBoost. Mỗi lượt chỉ thêm nhóm đặc trưng được mô tả bên dưới.

## Tạo đặc trưng

Lấy `channel` và `time_of_day_hour` từ giao dịch. Tính `hour_block_6h = time_of_day_hour // 6` để có bốn khung **00:00–05:59**, **06:00–11:59**, **12:00–17:59** và **18:00–23:59**. Ghép kênh với khung giờ rồi biểu diễn mỗi tổ hợp bằng một cột 0/1 (one-hot). Giao dịch POS lúc 04:00 bật cột POS × 00:00–05:59.

Với 5 kênh gốc và 4 khung giờ, nhóm bổ sung gồm **20 cột**. Cấu hình đầy đủ dùng **129 cột**.

## Huấn luyện và đánh giá mỗi fold

Giữ validation ngoài (V) để chấm điểm. Trong train ngoài (T), dùng phần A để fit và phần E để chọn số cây. Sau đó huấn luyện lại trên toàn bộ T với số cây đã chọn và dự đoán V cùng test. Danh mục kênh dùng để mã hóa được lấy từ phần huấn luyện ở mỗi lần fit. Khung giờ dùng bốn mốc cố định.

OOF là tập dự đoán ngoài fold, gồm một điểm dự đoán cho mỗi giao dịch train từ mô hình chưa học dòng đó.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

VARIANT = "CHANNEL-6H"
NAME = f"XGB-D03-{VARIANT}"

MODE = "view"  # "view", "smoke" hoặc "full"
VIEW_RESULT = "full"  # Kết quả hiển thị khi MODE = "view"

if MODE not in {"view", "smoke", "full"} or VIEW_RESULT not in {"smoke", "full"}:
    raise ValueError("Chọn MODE là view, smoke hoặc full và VIEW_RESULT là smoke hoặc full.")

working_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / "train.csv").is_file() and (path / "eda_review").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc một thư mục con của dự án.")

ablation_root = project_root / "eda_review" / "feature_ablation"
result_set = VIEW_RESULT if MODE == "view" else MODE
metrics_path = ablation_root / "results" / result_set / NAME / "metrics.json"
baseline_path = project_root / "s6e11_gpu" / "plan_runs" / "results" / "full" / "XGB-D03" / "metrics.json"

## Chạy trên RTX 4060

Mặc định `MODE = "view"` đọc kết quả đã lưu. Đổi sang `"smoke"` để chạy thử hai fold trên dữ liệu rút gọn hoặc `"full"` để chạy đủ năm fold. Khi chạy, dùng kernel Python của môi trường CUDA đã thiết lập cho dự án.

Lệnh tương đương từ thư mục dự án

```powershell
python eda_review/feature_ablation/run_ablation.py --variants CHANNEL-6H
```

In [2]:
import os
import subprocess
import sys

if MODE != "view":
    runner = ablation_root / "run_ablation.py"
    if not runner.is_file():
        raise FileNotFoundError(runner)

    command = [sys.executable, "-u", str(runner), "--variants", VARIANT]
    if MODE == "smoke":
        command.append("--smoke")

    with subprocess.Popen(
        command,
        cwd=project_root,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="")
        exit_code = process.wait()
        if exit_code != 0:
            raise subprocess.CalledProcessError(exit_code, command)

## Điểm số và số cây

Bảng đầu ghi ROC AUC và Average Precision (AP) trên các dự đoán OOF. Bảng tiếp theo ghi AUC, số cây và chênh lệch theo fold. Phép so với `XGB-D03` gốc dùng kết quả đủ dữ liệu.

In [3]:
if not metrics_path.is_file():
    print(f"Chưa có bảng điểm {result_set} cho {NAME}.")
else:
    candidate = json.loads(metrics_path.read_text(encoding="utf-8"))
    if candidate["smoke"] != (result_set == "smoke"):
        raise ValueError("Loại kết quả trong metrics.json không khớp thư mục đã chọn.")

    baseline = None
    if result_set == "full" and baseline_path.is_file():
        baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
        for field in ("rows", "test_rows", "fold_hash", "early_stop_hash", "inputs"):
            if candidate[field] != baseline[field]:
                raise ValueError(f"Đối chứng và mô hình mới khác nhau ở {field}.")

    items = [baseline, candidate] if baseline is not None else [candidate]
    totals = pd.DataFrame(
        [
            {
                "Mô hình": item["name"],
                "OOF ROC AUC": item["oof_auc"],
                "AP": item["average_precision"],
                "Log loss": item["log_loss"],
            }
            for item in items
        ]
    )
    if baseline is not None:
        totals["Δ OOF ROC AUC"] = totals["OOF ROC AUC"] - baseline["oof_auc"]
    display(totals.set_index("Mô hình").round(6))

    baseline_folds = {record["fold"]: record for record in baseline["folds"]} if baseline else {}
    if baseline and set(baseline_folds) != {record["fold"] for record in candidate["folds"]}:
        raise ValueError("Hai lượt chạy có danh sách fold khác nhau.")

    fold_rows = []
    for record in candidate["folds"]:
        row = {
            "Fold": record["fold"],
            "AUC bổ sung": record["auc"],
            "Số cây bổ sung": record["rounds"],
        }
        if baseline_folds:
            original = baseline_folds[record["fold"]]
            row.update({
                "AUC gốc": original["auc"],
                "Số cây gốc": original["rounds"],
                "Δ AUC": record["auc"] - original["auc"],
            })
        fold_rows.append(row)
    fold_summary = pd.DataFrame(fold_rows).set_index("Fold").sort_index()
    display(fold_summary.round(6))

    devices = sorted({record["device"] for record in candidate["folds"]})
    early_devices = sorted({record["early"]["device"] for record in candidate["folds"] if record.get("early")})
    print("Thiết bị huấn luyện trên T", ", ".join(devices))
    if early_devices:
        print("Thiết bị chọn số cây", ", ".join(early_devices))
    if baseline is not None:
        fold_wins = int((fold_summary["Δ AUC"] > 0).sum())
        print(f"AUC cao hơn đối chứng ở {fold_wins}/{len(fold_summary)} fold.")

Thiết bị huấn luyện trên T cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
Thiết bị chọn số cây cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
AUC cao hơn đối chứng ở 1/5 fold.


,OOF ROC AUC,AP,Log loss,Δ OOF ROC AUC
Mô hình,,,,
XGB-D03,0.950182,0.528763,0.044422,0.000000
XGB-D03-CHANNEL-6H,0.950108,0.528639,0.044431,-0.000074


,AUC bổ sung,Số cây bổ sung,AUC gốc,Số cây gốc,Δ AUC
Fold,,,,,
1,0.949775,2047,0.949852,2194,-0.000076
2,0.950873,1511,0.950884,1536,-0.000011
3,0.955007,1751,0.954957,1771,0.000050
4,0.950798,3001,0.950990,2306,-0.000193
5,0.944510,2170,0.944603,2131,-0.000093


## Đọc kết quả

Mốc sàng lọc của dự án là tăng ít nhất **0.000100 OOF ROC AUC** và có AUC cao hơn ở **ít nhất 4/5 fold**. Xem thêm AP để biết thứ hạng giao dịch gian lận ở đầu danh sách thay đổi thế nào. Các chỉ số trong notebook được tính từ tập train và phần validation của từng fold.

[Xem báo cáo tổng hợp hai lượt thử](../results/ket_qua_thu_dac_trung.md)